# Uncertainty Estimation Overview

Error bars are model-dependent estimates, not a property of the data. This notebook re-runs [`01_basic_fitting`](../01_basic_fitting/example.ipynb) and works through the **three tiers of uncertainty estimation** trspecfit exposes. The tiers are ordered by computational cost: each one assumes less and is more robust than the last, but takes longer to run. Usually you want the cheapest tier that is still valid for your case.

| Tier | Access via | Comes from | Cost | Characteristic failure |
|------|------------|------------|------|------------------------|
| #1 | `stderr` | covariance matrix at the fit minimum | free | correlated parameters, non-quadratic χ² surface |
| #2 | `try_ci=1` | F-test profiling (`lmfit.conf_interval`) | seconds | parameters at/near bounds |
| #3 | `mc_settings` | MCMC posterior sampling (`lmfit.emcee`) | minutes | unconverged chains |

Notebook 01's data is synthetic and ships with truth YAMLs, so we can check the error bars against the known answer.

Workflow:

0. **Re-run example 01**: This example uses the same synthetic data
1. **Estimate the noise σ**: σ is needed to compute χ²_red. χ²_red ≈ 1 is a quality-of-fit indicator.
2. **Create and review all three uncertainty estimates**: Run a baseline fit with `stderr` (free), `try_ci=1` (profiled CIs), and MCMC (`mc_settings`). Read fit results back with the `get_*` accessors — `get_fit_results` / `get_correlations` (tier 1), `get_conf_intervals` (tier 2), `get_mcmc` (tier 3) — and walk through the assumptions behind each tier.
3. **Compare each result against the known truth**: where tiers 1 and 2 show their failure modes.
4. **Scale up to the 2D fit**: a deliberately short chain to show tier 3's failure mode.

## 0. Re-run Example Notebook 01

In [ ]:
%%capture
# %%capture swallows notebook 01's plots/output

# %cd into notebook 01's dir so its project.yaml + data/models resolve
%cd -q "../01_basic_fitting"
# %run executes that notebook in the current kernel (variables stay live)
%run example.ipynb
# %cd - returns here
%cd -q -

In [ ]:
# Sanity check: confirm notebook 01's pipeline landed in this kernel.
n_free = sum(p.vary for p in file.model_2d.lmfit_pars.values())
print(f"file:                   {file.name}")
print(f"baseline model name:    {file.model_base.name}")
print(f"2D model name:          {file.model_2d.name}  ({n_free} free parameters)")

In [ ]:
from pathlib import Path

import corner
import numpy as np
import pandas as pd
from ruamel.yaml import YAML

from trspecfit.utils.lmfit import MC

In [ ]:
# helper: 1-sigma half-width per parameter from a CI / quantile table
def one_sigma_halfwidth(ci_df):
    half = (ci_df['+1.0'] - ci_df['-1.0']) / 2
    return dict(zip(ci_df['par[v]/sigma[>]'], half))

## 1. Noise Level as Validity Gate

All three tiers quantify parameter scatter *around the model you chose*. If the model is wrong, the error bars just describe scatter around the wrong answer. A good fit should leave residuals that are noise-sized, which the **σ-calibrated χ²_red ≈ 1** makes quantitative. See [`10_model_comparison`](../10_model_comparison/example.ipynb) for more on the σ machinery.

The noise level can be estimated without a model: in the pre-trigger block nothing evolves, so the per-pixel standard deviation over time is pure noise. `file.set_sigma(...)` registers it, and every fit from here on reports the calibrated χ²_red in its results.

This example simulates a photon-counting experiment, so σ grows with intensity. A single scalar σ is then only an average, and χ²_red lands near 1.3 rather than exactly 1. (Per-pixel σ is on the roadmap; it doesn't change the calibration logic.)

In [ ]:
# Model-free noise estimate from the pre-trigger block (nothing evolves there,
# so the per-pixel std over time is pure noise). Register it as the gate sigma.
pre_block = file.data[file.time < -10]
n_avg = pre_block.shape[0]
sigma_est = pre_block.std(axis=0, ddof=1).mean()
file.set_sigma(sigma_est)  # applies to all future fits on this file
print(f"sigma_est  = {sigma_est:.3f}  (per time slice)")

In [ ]:
# The baseline spectrum averages n_avg slices, so ITS noise is smaller by
# sqrt(n_avg) — the right sigma_ini when we sample the baseline posterior.
sigma_base = sigma_est / np.sqrt(n_avg)
print(f"sigma_base = {sigma_base:.3f}  (baseline = average of {n_avg} slices)")

## 2. One Fit, All Tiers

In practice you would start with the cheapest estimate and move to a more expensive tier only if you need to. Here we run a single baseline fit that turns on all three at once:

- `stderr` — free, from the `leastsq` covariance (always computed);
- `try_ci=1` — profiled confidence intervals (`lmfit.conf_interval`);
- `mc_settings` — MCMC posterior sampling (`lmfit.emcee`).

The MCMC settings that matter:

- `nwalkers` ≥ 2 × (free parameters + 1). The +1 is `__lnsigma`: with `is_weighted=False`, emcee samples the noise scale itself as a nuisance parameter, so the posterior widths self-calibrate instead of trusting our σ.
- `sigma_ini` sets the starting noise scale. The baseline averages `n_avg` slices, so its noise is `sigma_base = sigma_est/√n_avg`. Start it far off and the walkers spend their burn-in just finding the scale.
- `steps` has to beat the chain's autocorrelation time τ (≥ 50×τ; emcee measures τ and warns if the chain is too short). `burn` discards the walk-in.

Everything below reads off this single fit through the `get_*` accessors; nothing is re-fit. The fit prints a lot at once (the *Bound reached* warning, the emcee progress bar, the corner and acceptance plots), and the next sections take it one tier at a time.

In [ ]:
mc = MC(
    use_mc=1,
    nwalkers=20,      # >= 2 * (6 free + __lnsigma)
    steps=6000,       # > 50x the measured autocorrelation time (~80 here)
    burn=600,         # discard the walk-in
    thin=1,
    sigma_ini=sigma_base,  # start the sampled noise scale at the known level
)

In [ ]:
# one fit, all three methods: leastsq stderr + profiled CIs + MCMC posterior
file.fit_baseline(model_name='base', stages=2, try_ci=1, mc_settings=mc)

### 2.1. Tier 1 — `stderr`: the Free Error Bar

Every `leastsq` fit reports a `stderr` per parameter, from the covariance matrix (inverse Hessian) at the minimum. It carries two assumptions:

1. **The χ² surface is quadratic near the minimum.** Always true *close enough* to the minimum; the question is whether ±1σ is still *close*.
2. **The noise scale is right.** An unweighted fit can't know σ, so lmfit rescales the covariance to force χ²_red = 1 — that is, it estimates σ from the fit's own residuals, which only holds if the model is correct. That circularity is why the gate in section 1 comes first.

`stderr` is also **conditional**: it tells you how well a parameter is determined *if its correlated partners cooperate*. `get_correlations` shows which parameters are correlated; when they are strongly correlated, the real uncertainty is larger than `stderr` suggests ([`04_parameter_profiles`](../04_parameter_profiles/example.ipynb) has the extreme case: a tiny `stderr` on a degeneracy ridge).

In [ ]:
# stderr is the free byproduct of the leastsq covariance (one row per par)
file.get_fit_results(fit_type='baseline')[['name', 'value', 'stderr', 'vary']]

In [ ]:
# stderr is conditional: the correlation matrix shows who is entangled
file.get_correlations(fit_type='baseline').round(2)

### 2.2. Tier 2 — `try_ci=1`: Profiled Confidence Intervals

`try_ci=1` runs `lmfit.conf_interval`: it walks each parameter away from the optimum, re-optimizing all the others, until the fit degrades past an F-test threshold. No quadratic assumption, and the intervals come out asymmetric, reported at the levels in `ci_sigmas` (default `[1, 2, 3]`).

The fit above printed a `Bound reached` warning, and that is tier 2's failure mode. The Lorentzian fraction `GLP_01_m` sits against its hard lower bound at 0, so its −3σ excursion runs into the bound: lmfit warns `Bound reached with prob(GLP_01_m=0) ... < max(sigmas)` and reports `-inf` for that entry. A profiled CI isn't defined past a parameter bound. This is what the `try_ci=0  # see 12_uncertainty_mcmc` comments in the other notebooks were avoiding, and the MCMC posterior in the next section handles the same bound without breaking.

In [ ]:
# tier 2: the profiled CI table (note the -inf where GLP_01_m hit its 0 bound)
ci_table = file.get_conf_intervals(fit_type='baseline')
ci_table

### 2.3. Tier 3 — MCMC: Reading the Posterior

MCMC isn't a fit; it samples the posterior around the converged solution, so correlations, asymmetries, and pile-ups at bounds all show up directly. `get_mcmc` returns the sampled chain, the per-walker acceptance fractions, and the posterior quantile table.

- **Acceptance fraction** ≈ 0.2–0.5 means the walkers are mixing well.
- **The sampled noise scale** (`__lnsigma`) should match the model-free σ from section 1 — the posterior calibrates it rather than trusting our estimate.
- **The corner plot** (redrawn below from the chain) shows the pairwise marginals: tilted ellipses are correlations, and `GLP_01_m` piles up against its 0 bound instead of returning tier 2's `-inf`.

In [ ]:
mcmc = file.get_mcmc(fit_type='baseline')

# chain health + self-calibrated noise scale (should match sigma_base)
print(f"acceptance fraction: {mcmc.acceptance_fraction.mean():.2f}  (0.2-0.5)")
sigma_sampled = float(np.exp(mcmc.flatchain['__lnsigma']).median())
print(f"sampled noise scale: {sigma_sampled:.3f}  (sigma_base = {sigma_base:.3f})")

In [ ]:
# tier 3: the posterior quantile table
mcmc.table

In [ ]:
# corner plot regenerated from the chain (it also appeared at the fit above)
corner.corner(mcmc.flatchain, labels=list(mcmc.flatchain.columns));

## 3. The Three Tiers Side by Side — Checked Against the Truth

One row per free parameter: the ±1σ half-widths from all three tiers, the truth value from `data/models_energy_truth.yaml`, and the **pull** `(fit − truth)/stderr`.

Two things to check:

1. **Where the posterior is Gaussian and the bounds are far away, all three tiers agree** — `stderr`, the profiled CI, and the MCMC quantiles give the same ±1σ. When that happens, the cheapest tier is all you need.
2. **The pulls are O(1).** All six parameters land within about ±1σ of the truth, so the error bars are *calibrated*: they cover the true values at the rate they claim, not merely self-consistently. (One six-parameter draw is a spot check, not a proof, but a pull of 5 would tell you something is off.)

In [ ]:
# truth values, read via ruamel (the project's YAML stack, same as load_model)
# so scientific-notation scalars like 8E-2 parse as floats.
t = YAML(typ='safe').load(
    Path('../01_basic_fitting/data/models_energy_truth.yaml')
)['base']
truth = {f'LinBack_{k}': v[0] for k, v in t['LinBack'].items()}
truth |= {f'GLP_01_{k}': v[0] for k, v in t['GLP'].items()}

In [ ]:
# one row per free model parameter: each tier's 1-sigma half-width, the truth,
# and the pull. All three tiers come from the one baseline fit.
fit_df = file.get_fit_results(fit_type='baseline').set_index('name')
ci_1s = one_sigma_halfwidth(ci_table)
mcmc_1s = one_sigma_halfwidth(mcmc.table)
rows = [
    {
        'parameter': name,
        'truth': truth[name],
        'fit': row['value'],
        'stderr': row['stderr'],
        'ci_1sigma': ci_1s[name],
        'mcmc_1sigma': mcmc_1s[name],
        'pull': (row['value'] - truth[name]) / row['stderr'],
    }
    for name, row in fit_df.iterrows()
    if row['vary'] and name in truth
]
pd.DataFrame(rows).round(4)

## 4. Scaling Up: MCMC on the 2D Fit — and Its Convergence Diagnostics

The same call works on the global 2D fit; only the cost changes, since every posterior sample now evaluates the full 2D model. To keep the notebook fast we run a **deliberately short chain** and use it to read the convergence diagnostics:

- emcee prints *"The chain is shorter than 50 times the integrated autocorrelation time"* and lists τ per parameter — to trust the chain you'd need `steps ≥ 50×τ` (several thousand here).
- An unconverged chain shows up as **posterior quantiles much wider than `stderr`**: the walkers are still spreading out and haven't settled onto the posterior yet.

Note `sigma_ini=sigma_est` here (the per-slice noise, not averaged): the 2D fit sees the raw data, not the averaged baseline.

In [ ]:
mc_2d = MC(
    use_mc=1,
    nwalkers=20,
    steps=500,             # deliberately short — watch the diagnostics
    burn=100,
    thin=1,
    sigma_ini=sigma_est,   # per-slice noise: the 2D fit sees unaveraged data
)

In [ ]:
# same call, global 2D model — every posterior sample evaluates the full 2D fit.
file.fit_2d(model_name='2D', stages=1, try_ci=0, mc_settings=mc_2d)

In [ ]:
mcmc_2d = file.get_mcmc(fit_type='2d')
print(f"acceptance fraction: {mcmc_2d.acceptance_fraction.mean():.2f}  (0.2-0.5)")
sigma_2d = float(np.exp(mcmc_2d.flatchain['__lnsigma']).median())
print(f"sampled noise scale: {sigma_2d:.3f}  (sigma_est = {sigma_est:.3f})")

In [ ]:
# unconverged-chain symptom: MCMC 1-sigma half-widths well above stderr.
stderr_2d = file.get_fit_results(fit_type='2d').set_index('name')['stderr']
mcmc_2d_1s = one_sigma_halfwidth(mcmc_2d.table)
for name in ('GLP_01_x0', 'GLP_01_x0_expFun_01_tau'):
    print(
        f"{name}: stderr {stderr_2d[name]:.4f}  vs  "
        f"MCMC 1-sigma {mcmc_2d_1s[name]:.4f}"
    )

In [ ]:
# Validity gate for the 2D model (sigma-calibrated chi2_red ~ 1.2).
file.compare_models(fit_type='2d')[
    ['model', 'fit_type', 'chi2_red_raw', 'sigma_eff', 'chi2_red']
].tail(1)

## Tips for Uncertainty Estimation

**Before any error bar**
- Check the gate: σ-calibrated `chi2_red ≈ 1` (register σ with `file.set_sigma(...)`). An error bar on a model with structured residuals describes scatter around the wrong answer.

**One fit, all tiers**
- A single `fit_baseline(stages=2, try_ci=1, mc_settings=mc)` produces all three tiers; read them back with `get_fit_results` / `get_correlations` (tier 1), `get_conf_intervals` (tier 2), and `get_mcmc` (tier 3).

**Tier 1 — `stderr`**
- Free and usually right, *when* the posterior is Gaussian, bounds are far, and correlations are mild. Check `get_correlations(...)` before trusting it.

**Tier 2 — `try_ci=1`**
- `ci_sigmas=[1, 2, 3]` sets the reported levels (in σ units, not the noise σ).
- A *Bound reached* warning or `-inf` entry means the profiled interval ran into a parameter bound — undefined there, not zero. Switch to MCMC for those parameters.
- Needs ≥ 2 varying parameters (hence `try_ci=0` in single-parameter SbS fits).

**Tier 3 — MCMC**
- `nwalkers ≥ 2 × (n_free + 1)`; the +1 is the sampled noise scale `__lnsigma`.
- Set `sigma_ini` to the known/estimated noise level, and remember averaged data (baselines) has σ/√N_avg.
- Chain length: `steps ≥ 50×τ` (emcee prints τ and warns); acceptance 0.2–0.5; `burn` ≈ 10% of steps. Production posteriors: `steps=5000+`.
- Read the corner plot to diagnose: tilted ellipses are correlations; a pile-up at a bound is where tier 2 returned `-inf`.

## Next Steps

- σ machinery, `chi2_red` calibration, and model selection: [`10_model_comparison`](../10_model_comparison/example.ipynb)
- Persisting results — `conf_ci` tables and MCMC chains ride along in saved slots: [`11_save_load_export`](../11_save_load_export/example.ipynb)
- The underlying fit workflow: [`01_basic_fitting`](../01_basic_fitting/example.ipynb)